# 02_benchmark_ml20m_full_positive

- treat all ratings as positive interactions
- keep users with at least 5 interactions
- no rating threshold in the main benchmark track
- leave-one-out split
- stronger SASRec baseline
- same coding style and same function structure as the earlier notebook

## 0. Setup

In [14]:
import math
import time
import random
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [15]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

DATA_DIR = Path("../data/ml-20m")
PROC_DIR = Path("../data/processed_benchmark")
MODEL_DIR = Path("../models")
RESULT_DIR = Path("../reports/results")

PROC_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

MAX_LEN = 200
D_MODEL = 256
N_HEADS = 4
N_LAYERS = 4
DROPOUT = 0.1
BATCH_SIZE = 256
LR = 1e-3
WEIGHT_DECAY = 1e-5
EPOCHS = 20
EARLY_STOPPING_PATIENCE = 4

device: cuda


## 1. Load raw MovieLens 20M data

In [16]:
ratings = pd.read_csv(DATA_DIR / "ratings.csv")
movies = pd.read_csv(DATA_DIR / "movies.csv")
tags = pd.read_csv(DATA_DIR / "tags.csv")
genome_tags = pd.read_csv(DATA_DIR / "genome-tags.csv")
genome_scores = pd.read_csv(DATA_DIR / "genome-scores.csv")
links = pd.read_csv(DATA_DIR / "links.csv")

ratings["timestamp"] = pd.to_datetime(ratings["timestamp"], unit="s", errors="coerce")
tags["timestamp"] = pd.to_datetime(tags["timestamp"], unit="s", errors="coerce")

movies["year"] = movies["title"].str.extract(r"\((\d{4})\)").astype(float)
movies["clean_title"] = movies["title"].str.replace(r"\s*\(\d{4}\)$", "", regex=True)
movies["genres_list"] = movies["genres"].fillna("(no genres listed)").str.split("|")

tags = tags.dropna(subset=["tag"]).copy()
tags["tag"] = tags["tag"].astype(str).str.strip()
tags = tags[tags["tag"] != ""].copy()

print("ratings shape:", ratings.shape)
print("movies shape:", movies.shape)
print("tags shape:", tags.shape)
print("genome_tags shape:", genome_tags.shape)
print("genome_scores shape:", genome_scores.shape)
print("links shape:", links.shape)

print("\nratings time range:", ratings["timestamp"].min(), "->", ratings["timestamp"].max())
print("tags time range   :", tags["timestamp"].min(), "->", tags["timestamp"].max())

ratings shape: (20000263, 4)
movies shape: (27278, 6)
tags shape: (465541, 4)
genome_tags shape: (1128, 2)
genome_scores shape: (11709768, 3)
links shape: (27278, 3)

ratings time range: 1995-01-09 11:46:44 -> 2015-03-31 06:40:02
tags time range   : 2005-12-24 13:00:10 -> 2015-03-31 03:09:12


## 2. Benchmark preprocessing

This notebook uses the benchmark track:
- all ratings are positive interactions
- keep the last interaction if duplicate user-movie pairs exist
- keep users with at least 5 interactions

In [17]:
pair_counts = ratings.groupby(["userId", "movieId"]).size()

print("total rating rows:", len(ratings))
print("unique user-movie pairs:", len(pair_counts))
print("pairs with repeats:", (pair_counts > 1).sum())
print("rows inside repeated pairs:", pair_counts[pair_counts > 1].sum())
print("max repeats for one pair:", pair_counts.max())

total rating rows: 20000263
unique user-movie pairs: 20000263
pairs with repeats: 0
rows inside repeated pairs: 0
max repeats for one pair: 1


In [18]:
ratings_last = (
    ratings
    .sort_values(["userId", "movieId", "timestamp"])
    .drop_duplicates(["userId", "movieId"], keep="last")
    .reset_index(drop=True)
)

print("rows after last-rating dedup:", len(ratings_last))
print("users:", ratings_last["userId"].nunique())
print("items:", ratings_last["movieId"].nunique())

rows after last-rating dedup: 20000263
users: 138493
items: 26744


In [19]:
def kcore_filter_users_only(df, user_min=5):
    df = df.copy()

    while True:
        n_before = len(df)

        good_users = df["userId"].value_counts()
        good_users = good_users[good_users >= user_min].index
        df = df[df["userId"].isin(good_users)].copy()

        if len(df) == n_before:
            break

    return df

interactions = kcore_filter_users_only(ratings_last, user_min=5)
interactions = interactions.sort_values(["userId", "timestamp"]).reset_index(drop=True)

print("final benchmark interactions")
print("rows:", len(interactions))
print("users:", interactions["userId"].nunique())
print("items:", interactions["movieId"].nunique())

user_counts = interactions.groupby("userId").size()
item_counts = interactions.groupby("movieId").size()

print("\nuser count quantiles")
print(user_counts.quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).round(2))

print("\nitem count quantiles")
print(item_counts.quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).round(2))

final benchmark interactions
rows: 20000263
users: 138493
items: 26744

user count quantiles
0.25      35.00
0.50      68.00
0.75     155.00
0.90     334.00
0.95     520.00
0.99    1113.08
dtype: float64

item count quantiles
0.25        3.00
0.50       18.00
0.75      205.00
0.90     1305.70
0.95     3612.95
0.99    14388.69
dtype: float64


## 3. User split and ID mapping

In [20]:
user_ids = np.sort(interactions["userId"].unique())
movie_ids = np.sort(interactions["movieId"].unique())

user2idx = {u: i for i, u in enumerate(user_ids)}
item2idx = {m: i + 1 for i, m in enumerate(movie_ids)}
idx2item = {i: m for m, i in item2idx.items()}

interactions["user_idx"] = interactions["userId"].map(user2idx)
interactions["item_idx"] = interactions["movieId"].map(item2idx)

print("num users:", len(user2idx))
print("num items:", len(item2idx))
print(interactions[["userId", "movieId", "user_idx", "item_idx", "rating", "timestamp"]].head(10).to_string(index=False))

num users: 138493
num items: 26744
 userId  movieId  user_idx  item_idx  rating           timestamp
      1      924         0       908     3.5 2004-09-10 03:06:38
      1      919         0       903     3.5 2004-09-10 03:07:01
      1     2683         0      2598     3.5 2004-09-10 03:07:30
      1     1584         0      1533     3.5 2004-09-10 03:07:36
      1     1079         0      1058     4.0 2004-09-10 03:07:45
      1      653         0       646     3.0 2004-09-10 03:08:11
      1     2959         0      2874     4.0 2004-09-10 03:08:18
      1      337         0       334     3.5 2004-09-10 03:08:29
      1     1304         0      1276     3.0 2004-09-10 03:08:40
      1     3996         0      3903     4.0 2004-09-10 03:08:47


In [21]:
user_seq = (
    interactions.sort_values(["user_idx", "timestamp"])
    .groupby("user_idx")["item_idx"]
    .apply(list)
)

split_rows = []

for user_idx, seq in user_seq.items():
    split_rows.append({
        "user_idx": user_idx,
        "train_seq": seq[:-2],
        "val_seq": seq[:-2],
        "val_target": seq[-2],
        "test_seq": seq[:-1],
        "test_target": seq[-1],
        "full_len": len(seq)
    })

splits = pd.DataFrame(split_rows)

print("number of users in splits:", len(splits))
print("train seq length quantiles")
print(splits["train_seq"].apply(len).quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).round(2))
print("\nsplit preview")
print(splits.head(5).to_string(index=False))

number of users in splits: 138493
train seq length quantiles
0.25      33.00
0.50      66.00
0.75     153.00
0.90     332.00
0.95     518.00
0.99    1111.08
Name: train_seq, dtype: float64

split preview
 user_idx                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                train_seq                                                                                                                                                                                                          

In [22]:
interactions.to_pickle(PROC_DIR / "interactions_full_positive_u5.pkl")
splits.to_pickle(PROC_DIR / "splits_full_positive_u5.pkl")

pd.DataFrame({
    "userId": list(user2idx.keys()),
    "user_idx": list(user2idx.values())
}).to_pickle(PROC_DIR / "user_map_full_positive_u5.pkl")

pd.DataFrame({
    "movieId": list(item2idx.keys()),
    "item_idx": list(item2idx.values())
}).to_pickle(PROC_DIR / "item_map_full_positive_u5.pkl")

print("saved files in:", PROC_DIR)
print(sorted([p.name for p in PROC_DIR.iterdir()]))

saved files in: ../data/processed_benchmark
['interactions_full_positive_u5.pkl', 'item_map_full_positive_u5.pkl', 'splits_full_positive_u5.pkl', 'user_map_full_positive_u5.pkl']


## 4. Data loaders

In [23]:
interactions = pd.read_pickle(PROC_DIR / "interactions_full_positive_u5.pkl")
splits = pd.read_pickle(PROC_DIR / "splits_full_positive_u5.pkl")
item_map = pd.read_pickle(PROC_DIR / "item_map_full_positive_u5.pkl")
user_map = pd.read_pickle(PROC_DIR / "user_map_full_positive_u5.pkl")

num_users = len(user_map)
num_items = len(item_map)

print("num_users:", num_users)
print("num_items:", num_items)
print("splits:", splits.shape)

num_users: 138493
num_items: 26744
splits: (138493, 7)


In [24]:
def pad_seq(seq, max_len=200):
    seq = seq[-max_len:]
    out = np.zeros(max_len, dtype=np.int64)
    out[-len(seq):] = seq
    return out

class TrainDataset(Dataset):
    def __init__(self, splits_df, max_len=200):
        self.max_len = max_len
        self.seqs = splits_df["train_seq"].tolist()

        keep = []
        for seq in self.seqs:
            if len(seq) >= 3:
                keep.append(seq)

        self.data = keep

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        seq = self.data[idx]

        tokens = seq[:-1]
        targets = seq[1:]

        tokens = pad_seq(tokens, self.max_len)
        targets = pad_seq(targets, self.max_len)

        return {
            "seq": torch.tensor(tokens, dtype=torch.long),
            "target_seq": torch.tensor(targets, dtype=torch.long),
        }

class EvalDataset(Dataset):
    def __init__(self, splits_df, mode="val", max_len=200):
        self.max_len = max_len

        if mode == "val":
            self.seq_col = "val_seq"
            self.target_col = "val_target"
        else:
            self.seq_col = "test_seq"
            self.target_col = "test_target"

        self.seq = splits_df[self.seq_col].tolist()
        self.target = splits_df[self.target_col].tolist()

    def __len__(self):
        return len(self.seq)

    def __getitem__(self, idx):
        return {
            "seq": torch.tensor(pad_seq(self.seq[idx], self.max_len), dtype=torch.long),
            "target": torch.tensor(self.target[idx], dtype=torch.long),
        }

train_ds = TrainDataset(splits, max_len=MAX_LEN)
val_ds = EvalDataset(splits, mode="val", max_len=MAX_LEN)
test_ds = EvalDataset(splits, mode="test", max_len=MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print("train sequences:", len(train_ds))
print("val users:", len(val_ds))
print("test users:", len(test_ds))

train sequences: 138493
val users: 138493
test users: 138493


## 5. Base model

In [25]:
class SASRecBase(nn.Module):
    def __init__(self, num_items, max_len=200, d_model=256, n_heads=4, n_layers=4, dropout=0.1):
        super().__init__()

        self.num_items = num_items
        self.max_len = max_len
        self.d_model = d_model

        self.item_emb = nn.Embedding(num_items + 1, d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len + 1, d_model, padding_idx=0)

        self.emb_dropout = nn.Dropout(dropout)
        self.emb_norm = nn.LayerNorm(d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.final_norm = nn.LayerNorm(d_model)

        self._reset_parameters()

    def _reset_parameters(self):
        nn.init.normal_(self.item_emb.weight, std=0.02)
        nn.init.normal_(self.pos_emb.weight, std=0.02)

    def _make_pos_ids(self, seq):
        mask = (seq > 0).long()
        pos_ids = torch.cumsum(mask, dim=1) * mask
        pos_ids = pos_ids.clamp(max=self.max_len)
        return pos_ids

    def _causal_mask(self, L, device):
        return torch.triu(torch.ones(L, L, device=device, dtype=torch.bool), diagonal=1)

    def forward_sequence(self, seq):
        pos_ids = self._make_pos_ids(seq)

        x = self.item_emb(seq) + self.pos_emb(pos_ids)
        x = self.emb_norm(x)
        x = self.emb_dropout(x)

        pad_mask = (seq == 0)
        causal_mask = self._causal_mask(seq.size(1), seq.device)

        x = self.encoder(x, mask=causal_mask, src_key_padding_mask=pad_mask)
        x = self.final_norm(x)
        return x

    def encode(self, seq):
        x = self.forward_sequence(seq)
        lengths = (seq > 0).sum(dim=1).clamp(min=1)
        last_idx = lengths - 1
        h = x[torch.arange(seq.size(0), device=seq.device), last_idx]
        return h

    def predict_all(self, seq):
        h = self.encode(seq)
        logits = h @ self.item_emb.weight.T
        logits[:, 0] = -1e9
        return logits

    def predict_all_positions(self, seq):
        x = self.forward_sequence(seq)
        logits = x @ self.item_emb.weight.T
        logits[:, :, 0] = -1e9
        return logits

model = SASRecBase(
    num_items=num_items,
    max_len=MAX_LEN,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    dropout=DROPOUT
).to(device)

print(model)

/workspace/movie-transformer-recommender/.venv/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


SASRecBase(
  (item_emb): Embedding(26745, 256, padding_idx=0)
  (pos_emb): Embedding(201, 256, padding_idx=0)
  (emb_dropout): Dropout(p=0.1, inplace=False)
  (emb_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=1024, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=1024, out_features=256, bias=True)
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (final_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=Tru

## 6. Training loop

In [26]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    total_tokens = 0

    for batch in loader:
        seq = batch["seq"].to(device)
        target_seq = batch["target_seq"].to(device)

        optimizer.zero_grad()

        logits = model.predict_all_positions(seq)

        loss = nn.functional.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            target_seq.reshape(-1),
            ignore_index=0
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        valid_tokens = (target_seq > 0).sum().item()
        total_loss += loss.item() * valid_tokens
        total_tokens += valid_tokens

    return total_loss / max(total_tokens, 1)


@torch.no_grad()
def evaluate_topk(model, loader, device, k=10):
    model.eval()

    hits = 0.0
    ndcg = 0.0
    mrr = 0.0
    total = 0

    for batch in loader:
        seq = batch["seq"].to(device)
        target = batch["target"].to(device)

        logits = model.predict_all(seq)

        for i in range(seq.size(0)):
            seen = seq[i].unique()
            seen = seen[seen > 0]
            logits[i, seen] = -1e9

        topk = torch.topk(logits, k=k, dim=1).indices

        for i in range(seq.size(0)):
            tgt = target[i].item()
            pred = topk[i].tolist()

            if tgt in pred:
                rank = pred.index(tgt) + 1
                hits += 1.0
                ndcg += 1.0 / math.log2(rank + 1)
                mrr += 1.0 / rank

        total += seq.size(0)

    hr = hits / total
    ndcg = ndcg / total
    mrr = mrr / total

    return {
        f"HR@{k}": hr,
        f"NDCG@{k}": ndcg,
        f"MRR@{k}": mrr,
        f"Recall@{k}": hr,
    }

## 7. Train base benchmark model

In [27]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

history = []
best_val_ndcg = -1
best_state = None
bad_epochs = 0

train_start = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    t0 = time.perf_counter()

    train_loss = train_one_epoch(model, train_loader, optimizer, device)
    val_metrics = evaluate_topk(model, val_loader, device, k=10)

    epoch_time = time.perf_counter() - t0

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "epoch_sec": epoch_time,
        **val_metrics
    }
    history.append(row)

    print(
        f"epoch {epoch:02d} | "
        f"loss {train_loss:.4f} | "
        f"HR@10 {val_metrics['HR@10']:.4f} | "
        f"NDCG@10 {val_metrics['NDCG@10']:.4f} | "
        f"MRR@10 {val_metrics['MRR@10']:.4f} | "
        f"{epoch_time:.1f}s"
    )

    if val_metrics["NDCG@10"] > best_val_ndcg:
        best_val_ndcg = val_metrics["NDCG@10"]
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        bad_epochs = 0
    else:
        bad_epochs += 1

    if bad_epochs >= EARLY_STOPPING_PATIENCE:
        print(f"early stopping at epoch {epoch}")
        break

total_train_sec = time.perf_counter() - train_start
print("\ntraining finished in", round(total_train_sec, 2), "seconds")

OutOfMemoryError: CUDA out of memory. Tried to allocate 150.00 MiB. GPU 0 has a total capacity of 11.64 GiB of which 130.75 MiB is free. Process 1034814 has 8.92 GiB memory in use. Process 1283517 has 2.58 GiB memory in use. Of the allocated memory 2.37 GiB is allocated by PyTorch, and 86.80 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## 8. Evaluate and save outputs

In [ ]:
model.load_state_dict(best_state)

test_start = time.perf_counter()
test_metrics = evaluate_topk(model, test_loader, device, k=10)
test_sec = time.perf_counter() - test_start

print("best validation NDCG@10:", round(best_val_ndcg, 6))
print("test metrics:")
for k, v in test_metrics.items():
    print(k, ":", round(v, 6))
print("test eval seconds:", round(test_sec, 2))

history_df = pd.DataFrame(history)
history_df

In [ ]:
torch.save(best_state, MODEL_DIR / "sasrec_base_full_positive_u5.pt")
history_df.to_csv(RESULT_DIR / "sasrec_base_full_positive_u5_history.csv", index=False)

pd.DataFrame([{
    "model": "sasrec_base_full_positive_u5",
    "num_users": num_users,
    "num_items": num_items,
    "max_len": MAX_LEN,
    "d_model": D_MODEL,
    "n_heads": N_HEADS,
    "n_layers": N_LAYERS,
    "dropout": DROPOUT,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "epochs_planned": EPOCHS,
    "train_total_sec": total_train_sec,
    "test_eval_sec": test_sec,
    **test_metrics
}]).to_csv(RESULT_DIR / "sasrec_base_full_positive_u5_test_metrics.csv", index=False)

print("saved model:", MODEL_DIR / "sasrec_base_full_positive_u5.pt")
print("saved history:", RESULT_DIR / "sasrec_base_full_positive_u5_history.csv")
print("saved test metrics:", RESULT_DIR / "sasrec_base_full_positive_u5_test_metrics.csv")

## 9.  comparison
# Stronger base benchmar + Hybrid gated

In [28]:

#L — stronger benchmark config
# stronger benchmark config
MAX_LEN = 200
D_MODEL = 256
N_HEADS = 4
N_LAYERS = 4
DROPOUT = 0.1
BATCH_SIZE = 256
LR = 1e-3
WEIGHT_DECAY = 1e-5
EPOCHS = 20
GRAD_CLIP = 1.0

print("MAX_LEN:", MAX_LEN)
print("D_MODEL:", D_MODEL)
print("N_HEADS:", N_HEADS)
print("N_LAYERS:", N_LAYERS)
print("DROPOUT:", DROPOUT)
print("BATCH_SIZE:", BATCH_SIZE)
print("LR:", LR)
print("WEIGHT_DECAY:", WEIGHT_DECAY)
print("EPOCHS:", EPOCHS)

MAX_LEN: 200
D_MODEL: 256
N_HEADS: 4
N_LAYERS: 4
DROPOUT: 0.1
BATCH_SIZE: 256
LR: 0.001
WEIGHT_DECAY: 1e-05
EPOCHS: 20


In [29]:

# M — rebuild loaders with the stronger config
train_ds = TrainDataset(splits, max_len=MAX_LEN)
val_ds = EvalDataset(splits, mode="val", max_len=MAX_LEN)
test_ds = EvalDataset(splits, mode="test", max_len=MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print("train sequences:", len(train_ds))
print("val users:", len(val_ds))
print("test users:", len(test_ds))

train sequences: 138493
val users: 138493
test users: 138493


In [30]:

# N — stronger base model
base_model = SASRecBase(
    num_items=num_items,
    max_len=MAX_LEN,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    dropout=DROPOUT
).to(device)

print(base_model)

SASRecBase(
  (item_emb): Embedding(26745, 256, padding_idx=0)
  (pos_emb): Embedding(201, 256, padding_idx=0)
  (emb_dropout): Dropout(p=0.1, inplace=False)
  (emb_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=1024, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=1024, out_features=256, bias=True)
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (final_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=Tru

/workspace/movie-transformer-recommender/.venv/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [31]:

# O — optimizer and scheduler helpers
def build_optimizer(model, lr=1e-3, weight_decay=1e-5):
    return torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

def build_scheduler(optimizer, epochs):
    return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

def train_one_epoch(model, loader, optimizer, device, grad_clip=1.0):
    model.train()
    total_loss = 0.0
    total_tokens = 0

    for batch in loader:
        seq = batch["seq"].to(device)
        target_seq = batch["target_seq"].to(device)

        optimizer.zero_grad()

        logits = model.predict_all_positions(seq)

        loss = nn.functional.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            target_seq.reshape(-1),
            ignore_index=0
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()

        valid_tokens = (target_seq > 0).sum().item()
        total_loss += loss.item() * valid_tokens
        total_tokens += valid_tokens

    return total_loss / max(total_tokens, 1)


@torch.no_grad()
def evaluate_topk(model, loader, device, k=10):
    model.eval()

    hits = 0.0
    ndcg = 0.0
    mrr = 0.0
    total = 0

    for batch in loader:
        seq = batch["seq"].to(device)
        target = batch["target"].to(device)

        logits = model.predict_all(seq)

        for i in range(seq.size(0)):
            seen = seq[i].unique()
            seen = seen[seen > 0]
            logits[i, seen] = -1e9

        topk = torch.topk(logits, k=k, dim=1).indices

        for i in range(seq.size(0)):
            tgt = target[i].item()
            pred = topk[i].tolist()

            if tgt in pred:
                rank = pred.index(tgt) + 1
                hits += 1.0
                ndcg += 1.0 / math.log2(rank + 1)
                mrr += 1.0 / rank

        total += seq.size(0)

    hr = hits / total
    ndcg = ndcg / total
    mrr = mrr / total

    return {
        f"HR@{k}": hr,
        f"NDCG@{k}": ndcg,
        f"MRR@{k}": mrr,
        f"Recall@{k}": hr,
    }

In [32]:

# P — train the stronger benchmark base model
optimizer = build_optimizer(base_model, lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = build_scheduler(optimizer, epochs=EPOCHS)

history_base = []
best_val_ndcg_base = -1
best_state_base = None

train_start_base = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    t0 = time.perf_counter()

    train_loss = train_one_epoch(base_model, train_loader, optimizer, device, grad_clip=GRAD_CLIP)
    val_metrics = evaluate_topk(base_model, val_loader, device, k=10)

    scheduler.step()

    epoch_time = time.perf_counter() - t0
    lr_now = optimizer.param_groups[0]["lr"]

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "epoch_sec": epoch_time,
        "lr": lr_now,
        **val_metrics
    }
    history_base.append(row)

    print(
        f"epoch {epoch:02d} | "
        f"loss {train_loss:.4f} | "
        f"HR@10 {val_metrics['HR@10']:.4f} | "
        f"NDCG@10 {val_metrics['NDCG@10']:.4f} | "
        f"MRR@10 {val_metrics['MRR@10']:.4f} | "
        f"lr {lr_now:.6f} | "
        f"{epoch_time:.1f}s"
    )

    if val_metrics["NDCG@10"] > best_val_ndcg_base:
        best_val_ndcg_base = val_metrics["NDCG@10"]
        best_state_base = {k: v.cpu().clone() for k, v in base_model.state_dict().items()}

total_train_sec_base = time.perf_counter() - train_start_base
print("\nbase benchmark training finished in", round(total_train_sec_base, 2), "seconds")

OutOfMemoryError: CUDA out of memory. Tried to allocate 50.00 MiB. GPU 0 has a total capacity of 11.64 GiB of which 16.75 MiB is free. Process 1034814 has 8.92 GiB memory in use. Process 1283517 has 2.69 GiB memory in use. Of the allocated memory 2.52 GiB is allocated by PyTorch, and 47.55 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)